In [34]:
import pandas as pd
import numpy as np
import spacy

In [35]:
data=pd.read_csv('merchant_risk_dataset_100k.csv')

In [36]:
data.shape

(100000, 12)

In [37]:
data['Category'].unique()

array(['Travel', 'E-commerce', 'Crypto', 'Retail', 'Gaming', 'Services'],
      dtype=object)

In [38]:
data.describe()

,TransactionCount,TotalVolume,RefundRate,ChargebackRate,AvgTransactionValue,TxnGrowthRate
count,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.0,100000.0
mean,1211.670630,2.728546e+05,0.116867,0.081062,0.0,0.0
std,1236.776071,4.359344e+05,0.109295,0.081466,0.0,0.0
min,50.000000,1.018260e+03,0.000000,0.000000,0.0,0.0
25%,310.000000,1.776401e+04,0.031000,0.019000,0.0,0.0
50%,690.000000,7.965212e+04,0.081000,0.051000,0.0,0.0
75%,1707.000000,2.859723e+05,0.166000,0.113000,0.0,0.0
max,5000.000000,2.493028e+06,0.400000,0.300000,0.0,0.0


In [39]:

# Load only required components (IMPORTANT SPEED BOOST)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


# Keyword sets (faster than matcher for your use case)
HIGH = {"crypto", "cryptocurrency", "bitcoin", "trading", "exchange", "gaming", "betting", "casino", "gambling"}
MEDIUM = {"travel", "booking", "flight", "hotel", "tickets"}
LOW = {"retail", "store", "shopping", "services", "subscription"}

# Results
risks = []
matched_words = []

# 🚀 FAST processing
for doc in nlp.pipe(data["Description"].astype(str), batch_size=1000):
    tokens = {token.lemma_.lower() for token in doc}

    found = tokens & (HIGH | MEDIUM | LOW)

    if tokens & HIGH:
        risks.append("High")
    elif tokens & MEDIUM:
        risks.append("Medium")
    elif tokens & LOW:
        risks.append("Low")
    else:
        risks.append("Unknown")

    matched_words.append(", ".join(found))

# Add columns
data["NLP_Risk"] = risks
data["Matched_Words"] = matched_words

# Save
data.to_csv("merchant_risk_dataset_100k.csv", index=False)

print("✅ FAST NLP completed")

✅ FAST NLP completed


In [40]:
data.head(10)

,MerchantID,TransactionCount,TotalVolume,RefundRate,ChargebackRate,Category,Description,Month,NLP_Risk,Matched_Words,AvgTransactionValue,TxnGrowthRate
0,M1,280,9918.22,0.040,0.006,Travel,Flight and hotel booking services,2025-01-01,Medium,"booking, hotel, flight",0,0.0
1,M10,1663,756412.14,0.321,0.228,E-commerce,Online shopping platform selling consumer goods,2025-06-01,Low,shopping,0,0.0
2,M100,291,21850.36,0.032,0.027,Travel,Flight and hotel booking services,2025-08-01,Medium,"booking, hotel, flight",0,0.0
3,M1000,1108,191065.29,0.084,0.062,Crypto,Cryptocurrency exchange and trading platform,2025-08-01,High,"cryptocurrency, exchange, trading",0,0.0
4,M10000,139,13143.05,0.046,0.007,Retail,Physical and online retail store,2025-02-01,Low,"store, retail",0,0.0
5,M100000,486,39249.80,0.034,0.006,Retail,Physical and online retail store,2025-07-01,Low,"store, retail",0,0.0
6,M10001,3114,1455184.05,0.235,0.252,Travel,Flight and hotel booking services,2025-05-01,Medium,"booking, hotel, flight",0,0.0
7,M10002,279,7403.28,0.026,0.009,Travel,Flight and hotel booking services,2025-02-01,Medium,"booking, hotel, flight",0,0.0
8,M10003,1587,79821.54,0.101,0.077,Gaming,Online gaming and betting platform,2025-01-01,High,gaming,0,0.0
9,M10004,1744,95196.04,0.090,0.067,E-commerce,Online shopping platform selling consumer goods,2025-02-01,Low,shopping,0,0.0


In [55]:
# Extract numeric part from MerchantID
data["MerchantNum"] = data["MerchantID"].str.replace("M", "").astype(int)

# Sort properly
data = data.sort_values(by=["MerchantNum", "Month"])

# Optional: drop helper column
data.drop(columns=["MerchantNum"], inplace=True)

In [56]:


# ---------------- PREPROCESS MONTH ----------------
# Convert Month to datetime (VERY IMPORTANT)
data["Month"] = pd.to_datetime(data["Month"])

# Sort for correct growth calculation
data = data.sort_values(by=["MerchantID", "Month"])

# ---------------- FEATURE 1: AVG TRANSACTION ----------------
data["AvgTransactionValue"] = data["TotalVolume"] / data["TransactionCount"]

# Handle division issues
data["AvgTransactionValue"]=data["AvgTransactionValue"].replace([float("inf"), -float("inf")], 0)
data["AvgTransactionValue"]=data["AvgTransactionValue"].fillna(0)

# ---------------- FEATURE 2: TRANSACTION GROWTH ----------------

# Get previous month's transaction count for same merchant
data["PrevTxn"] = data.groupby("MerchantID")["TransactionCount"].shift(1)

# Growth calculation
data["TxnGrowthRate"] = (
    (data["TransactionCount"] - data["PrevTxn"]) / data["PrevTxn"]
)

# Handle first record (no previous month)
data["TxnGrowthRate"]=data["TxnGrowthRate"].fillna(0)

# Optional: clip extreme values (good for ML later)
data["TxnGrowthRate"] = data["TxnGrowthRate"].clip(-1, 5)

# ---------------- DROP HELPER COLUMN ----------------
data.drop(columns=["PrevTxn"], inplace=True)

# ---------------- SAVE BACK ----------------
data.to_csv("merchant_risk_dataset_100k.csv", index=False)

print("✅ Feature Engineering Completed")
print(data.head())

✅ Feature Engineering Completed
  MerchantID  TransactionCount  TotalVolume  RefundRate  ChargebackRate  \
0         M1               280      9918.22       0.040           0.006   
1        M10              1663    756412.14       0.321           0.228   
2       M100               291     21850.36       0.032           0.027   
3      M1000              1108    191065.29       0.084           0.062   
4     M10000               139     13143.05       0.046           0.007   

     Category                                      Description      Month  \
0      Travel                Flight and hotel booking services 2025-01-01   
1  E-commerce  Online shopping platform selling consumer goods 2025-06-01   
2      Travel                Flight and hotel booking services 2025-08-01   
3      Crypto     Cryptocurrency exchange and trading platform 2025-08-01   
4      Retail                 Physical and online retail store 2025-02-01   

  NLP_Risk                      Matched_Words  AvgTran

In [57]:
# FIX MerchantID sorting
data["MerchantNum"] = data["MerchantID"].str.replace("M", "").astype(int)
data = data.sort_values(by=["MerchantNum", "Month"])
data.drop(columns=["MerchantNum"], inplace=True)

In [58]:
data.head()

,MerchantID,TransactionCount,TotalVolume,RefundRate,ChargebackRate,Category,Description,Month,NLP_Risk,Matched_Words,AvgTransactionValue,TxnGrowthRate
0,M1,280,9918.22,0.040,0.006,Travel,Flight and hotel booking services,2025-01-01,Medium,"booking, hotel, flight",35.422214,0.0
11112,M2,58,5210.96,0.018,0.026,Services,Digital service provider and subscriptions,2025-11-01,Low,subscription,89.844138,0.0
22223,M3,306,19759.60,0.011,0.029,E-commerce,Online shopping platform selling consumer goods,2025-05-01,Low,shopping,64.573856,0.0
33334,M4,1245,235307.01,0.061,0.044,E-commerce,Online shopping platform selling consumer goods,2025-06-01,Low,shopping,189.001614,0.0
44445,M5,1489,186629.22,0.070,0.052,Services,Digital service provider and subscriptions,2025-02-01,Low,subscription,125.338630,0.0


In [70]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


# ---------------- SELECT FEATURES ----------------
features = [
    "TransactionCount",
    "TotalVolume",
    "RefundRate",
    "ChargebackRate",
    "AvgTransactionValue",
    "TxnGrowthRate"
]

X = data[features].copy()

# ---------------- FEATURE WEIGHTING ----------------
# Give more importance to risky behavior
X["RefundRate"] = X["RefundRate"] * 2
X["ChargebackRate"] = X["ChargebackRate"] * 2

# ---------------- SCALING ----------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------------- ISOLATION FOREST ----------------
model = IsolationForest(
    n_estimators=100,
    contamination=0.1,   # 5% anomalies
    random_state=42,
    n_jobs=-1             # use all CPU cores (faster)
)

# Fit model
model.fit(X_scaled)

# ---------------- PREDICTIONS ----------------
# -1 = anomaly, 1 = normal
preds = model.predict(X_scaled)

# Convert to 0 (normal) and 1 (anomaly)
data["Anomaly_Label"] = pd.Series(preds).map({1: 0, -1: 1})

# ---------------- ANOMALY SCORE ----------------
data["Anomaly_Score"] = model.decision_function(X_scaled)

# ---------------- SAVE BACK ----------------
data.to_csv("merchant_risk_dataset_100k.csv", index=False)

print("✅ Anomaly Detection Completed Successfully")
print(data[["Anomaly_Label", "Anomaly_Score"]].head())

✅ Anomaly Detection Completed Successfully
       Anomaly_Label  Anomaly_Score
0                  0       0.156189
11112              0       0.158741
22223              0       0.176252
33334              0       0.115095
44445              0       0.142399


In [71]:
data.head(20)

,MerchantID,TransactionCount,TotalVolume,RefundRate,ChargebackRate,Category,Description,Month,NLP_Risk,Matched_Words,AvgTransactionValue,TxnGrowthRate,Anomaly_Label,Anomaly_Score
0,M1,280,9918.22,0.040,0.006,Travel,Flight and hotel booking services,2025-01-01,Medium,"booking, hotel, flight",35.422214,0.0,0,0.156189
11112,M2,58,5210.96,0.018,0.026,Services,Digital service provider and subscriptions,2025-11-01,Low,subscription,89.844138,0.0,0,0.158741
22223,M3,306,19759.60,0.011,0.029,E-commerce,Online shopping platform selling consumer goods,2025-05-01,Low,shopping,64.573856,0.0,0,0.176252
33334,M4,1245,235307.01,0.061,0.044,E-commerce,Online shopping platform selling consumer goods,2025-06-01,Low,shopping,189.001614,0.0,0,0.115095
44445,M5,1489,186629.22,0.070,0.052,Services,Digital service provider and subscriptions,2025-02-01,Low,subscription,125.338630,0.0,0,0.142399
55556,M6,450,40146.63,0.027,0.022,E-commerce,Online shopping platform selling consumer goods,2025-07-01,Low,shopping,89.214733,0.0,0,0.163816
66667,M7,308,29842.63,0.034,0.028,E-commerce,Online shopping platform selling consumer goods,2025-05-01,Low,shopping,96.891656,0.0,0,0.177938
77778,M8,737,145161.72,0.150,0.072,Crypto,Cryptocurrency exchange and trading platform,2025-12-01,High,"cryptocurrency, exchange, trading",196.962985,0.0,0,0.121060
88889,M9,974,69089.75,0.085,0.096,Gaming,Online gaming and betting platform,2025-10-01,High,gaming,70.934035,0.0,0,0.128224
1,M10,1663,756412.14,0.321,0.228,E-commerce,Online shopping platform selling consumer goods,2025-06-01,Low,shopping,454.847949,0.0,0,0.014500


In [72]:
print(data["Anomaly_Label"].value_counts())

Anomaly_Label
0    90000
1    10000
Name: count, dtype: int64


In [73]:
total = len(data)

normal_pct = (data["Anomaly_Label"] == 0).mean() * 100
anomaly_pct = (data["Anomaly_Label"] == 1).mean() * 100

print(f"Normal: {normal_pct:.2f}%")
print(f"Anomaly: {anomaly_pct:.2f}%")

Normal: 90.00%
Anomaly: 10.00%


In [77]:
data[data["Anomaly_Label"] == 1].head(10)

,MerchantID,TransactionCount,TotalVolume,RefundRate,ChargebackRate,Category,Description,Month,NLP_Risk,Matched_Words,AvgTransactionValue,TxnGrowthRate,Anomaly_Label,Anomaly_Score,Cluster,Cluster_Risk
17779,M26,495,35303.08,0.016,0.005,Travel,Flight and hotel booking services,2025-02-01,Medium,"booking, hotel, flight",71.319354,0.0,1,0.140284,0,Low
23335,M31,2139,740843.41,0.350,0.155,Crypto,Cryptocurrency exchange and trading platform,2025-02-01,High,"cryptocurrency, exchange, trading",346.350355,0.0,1,0.043828,1,Medium
26668,M34,2850,1064252.48,0.387,0.194,Gaming,Online gaming and betting platform,2025-04-01,High,gaming,373.421923,0.0,1,0.023762,2,High
36668,M43,928,175441.09,0.056,0.083,Gaming,Online gaming and betting platform,2025-02-01,High,gaming,189.052899,0.0,1,0.121063,0,Low
37779,M44,4359,922276.98,0.153,0.287,Crypto,Cryptocurrency exchange and trading platform,2025-10-01,High,"cryptocurrency, exchange, trading",211.579945,0.0,1,-0.038766,2,High
45557,M51,710,138092.84,0.147,0.067,Gaming,Online gaming and betting platform,2025-04-01,High,gaming,194.496958,0.0,1,0.123983,0,Low
62223,M66,1353,135074.18,0.063,0.068,Travel,Flight and hotel booking services,2025-06-01,Medium,"booking, hotel, flight",99.833097,0.0,1,0.148726,0,Low
63334,M67,412,38772.15,0.037,0.011,Services,Digital service provider and subscriptions,2025-04-01,Low,subscription,94.107160,0.0,1,0.172178,0,Low
80001,M82,102,8574.95,0.002,0.025,Retail,Physical and online retail store,2025-03-01,Low,"store, retail",84.068137,0.0,1,0.133274,0,Low
86667,M88,1780,253783.50,0.113,0.066,Retail,Physical and online retail store,2025-04-01,Low,"store, retail",142.575000,0.0,1,0.149526,0,Low


In [74]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ---------------- FEATURES ----------------
features = [
    "TransactionCount",
    "TotalVolume",
    "RefundRate",
    "ChargebackRate",
    "AvgTransactionValue",
    "TxnGrowthRate"
]

X = data[features].copy()

# Same weighting (important)
X["RefundRate"] *= 2
X["ChargebackRate"] *= 2

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------------- KMEANS ----------------
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

data["Cluster"] = kmeans.fit_predict(X_scaled)

print("✅ Clustering Done")
print(data["Cluster"].value_counts())

✅ Clustering Done
Cluster
0    73410
1    16286
2    10304
Name: count, dtype: int64


In [76]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled, data["Cluster"])
print("Silhouette Score:", score)

Silhouette Score: 0.6039201830570909


In [75]:
# Find cluster risk by avg refund + chargeback
cluster_risk = data.groupby("Cluster")[["RefundRate", "ChargebackRate"]].mean()

# Sort clusters by risk
cluster_risk["RiskScore"] = cluster_risk["RefundRate"] + cluster_risk["ChargebackRate"]
cluster_risk = cluster_risk.sort_values("RiskScore")

# Map clusters
risk_map = {
    cluster_risk.index[0]: "Low",
    cluster_risk.index[1]: "Medium",
    cluster_risk.index[2]: "High"
}

data["Cluster_Risk"] = data["Cluster"].map(risk_map)

In [78]:
weights = {
    "RefundRate": 0.30,          # boosted (business critical)
    "ChargebackRate": 0.30,      # boosted
    "AvgTransactionValue": 0.15,
    "TotalVolume": 0.10,
    "TransactionCount": 0.10,
    "TxnGrowthRate": 0.05        # keep small (future use)
}

In [84]:
data["Risk_Score"] = (
    0.20 * data["RefundRate"] +
    0.15 * data["ChargebackRate"] +
    0.25 * data["AvgTransactionValue"] +
    0.05 * data["TxnGrowthRate"] +
    0.10 * data["Anomaly_Label"]
)

In [91]:
def assign_final_risk(score):
    if score > 0.6:
        return "High"
    elif score > 0.3:
        return "Medium"
    else:
        return "Low"

data["Final_Risk"] = data["Risk_Score"].apply(assign_final_risk)

# SAVE IT
data.to_csv("merchant_risk_dataset_100k.csv", index=False)

print("✅ Final_Risk added and saved")

✅ Final_Risk added and saved


In [92]:
def generate_reason(row):
    reasons = []

    if row["RefundRate"] > 0.15:
        reasons.append("High refund rate")

    if row["ChargebackRate"] > 0.1:
        reasons.append("High chargeback rate")

    if row["TxnGrowthRate"] > 1:
        reasons.append("Sudden transaction spike")

    if row["Anomaly_Label"] == 1:
        reasons.append("Anomalous behavior detected")

    if row["NLP_Risk"] == "High":
        reasons.append("High-risk category (NLP)")

    if not reasons:
        return "Normal behavior"

    return " + ".join(reasons)


data["Reason"] = data.apply(generate_reason, axis=1)

In [93]:
category_insight = data.groupby("Category")[["RefundRate", "ChargebackRate"]].mean()

print(category_insight)

            RefundRate  ChargebackRate
Category                              
Crypto        0.204570        0.145847
E-commerce    0.072572        0.048610
Gaming        0.205847        0.145909
Retail        0.072441        0.048745
Services      0.072774        0.048552
Travel        0.071403        0.047542


In [94]:
high_risk = data[data["Final_Risk"] == "High"]

print("Avg Refund:", high_risk["RefundRate"].mean())
print("Avg Chargeback:", high_risk["ChargebackRate"].mean())

Avg Refund: 0.11686685000000001
Avg Chargeback: 0.08106209


In [95]:
import joblib

joblib.dump(model, "models/isolation_forest.pkl")
joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

In [96]:
print(data["Risk_Score"].describe())

count    100000.000000
mean         36.449556
std          29.434464
min           5.005417
25%          15.919868
50%          24.619744
75%          45.794799
max         125.178232
Name: Risk_Score, dtype: float64


In [98]:
low = data["Risk_Score"].quantile(0.33)
high = data["Risk_Score"].quantile(0.66)

def assign_final_risk(score):
    if score > high:
        return "High"
    elif score > low:
        return "Medium"
    else:
        return "Low"

data["Final_Risk"] = data["Risk_Score"].apply(assign_final_risk)
data.to_csv("merchant_risk_dataset_100k.csv", index=False)